In [1]:
df_orders = spark.table("retail_lakehouse.dbo.bronze_orders")

display(df_orders)

StatementMeta(, 257476b0-b61f-4b40-ba71-f18eedd10e8e, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b92798f3-4a97-4afa-a72a-4f2320a6425f)

In [2]:
from pyspark.sql.functions import col

df_orders.select(
    col("order_id").isNull().alias("order_id_null"),
    col("customer_id").isNull().alias("customer_id_null"),
    col("product_id").isNull().alias("product_id_null"),
    col("quantity").isNull().alias("quantity_null"),
    col("order_date").isNull().alias("order_date_null")
).show()

StatementMeta(, 257476b0-b61f-4b40-ba71-f18eedd10e8e, 4, Finished, Available, Finished, False)

+-------------+----------------+---------------+-------------+---------------+
|order_id_null|customer_id_null|product_id_null|quantity_null|order_date_null|
+-------------+----------------+---------------+-------------+---------------+
|        false|           false|          false|        false|          false|
|        false|           false|          false|        false|          false|
|        false|           false|          false|        false|          false|
|        false|           false|          false|        false|          false|
|        false|           false|          false|        false|          false|
|        false|           false|          false|        false|          false|
|        false|           false|          false|        false|          false|
|        false|           false|          false|        false|          false|
+-------------+----------------+---------------+-------------+---------------+



In [3]:
from pyspark.sql.functions import count

df_orders.groupBy("order_id") \
    .agg(count("*").alias("count")) \
    .filter(col("count") > 1) \
    .show()

StatementMeta(, 257476b0-b61f-4b40-ba71-f18eedd10e8e, 5, Finished, Available, Finished, False)

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [4]:
df_orders.filter(col("quantity") <= 0).show()

StatementMeta(, 257476b0-b61f-4b40-ba71-f18eedd10e8e, 6, Finished, Available, Finished, False)

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
+--------+-----------+----------+--------+----------+



In [5]:
df_orders_silver = (
    df_orders
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("quantity") > 0)
    .dropDuplicates(["order_id"])
)

display(df_orders_silver)

StatementMeta(, 257476b0-b61f-4b40-ba71-f18eedd10e8e, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 18ba3383-e407-4b33-8521-25103d240c78)

In [6]:
df_orders_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("retail_lakehouse.dbo.silver_orders_pyspark")

StatementMeta(, 257476b0-b61f-4b40-ba71-f18eedd10e8e, 8, Finished, Available, Finished, False)

In [7]:
df_dataflow = spark.table("retail_lakehouse.dbo.silver_orders")
df_pyspark = spark.table("retail_lakehouse.dbo.silver_orders_pyspark")

print("Dataflow rows:", df_dataflow.count())
print("PySpark rows:", df_pyspark.count())

StatementMeta(, 257476b0-b61f-4b40-ba71-f18eedd10e8e, 9, Finished, Available, Finished, False)

Dataflow rows: 8
PySpark rows: 8


In [8]:
only_in_dataflow = df_dataflow.subtract(df_pyspark)
only_in_pyspark = df_pyspark.subtract(df_dataflow)

print("Nur in Dataflow-Version:", only_in_dataflow.count())
print("Nur in PySpark-Version:", only_in_pyspark.count())

StatementMeta(, 257476b0-b61f-4b40-ba71-f18eedd10e8e, 10, Finished, Available, Finished, False)

Nur in Dataflow-Version: 0
Nur in PySpark-Version: 0
